<center><h1>Huang_Shane_HW3</h1></center>
<br>
<br>

Name: Shane Huang
<br>
Github Username: shae513351
<br>
USC ID: 5886395143

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [2]:
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

import numpy as np
import pandas as pd

Get the AReM Data Set

In [3]:
# checking a dataset to find the ideal structure(shape,time...)
DATA_DIR = Path("../data/AReM")

COLUMNS = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

file_path = DATA_DIR/"bending1"/"dataset1.csv"

df = pd.read_csv(
    file_path,
    comment="#", #ignore metadata started with #
    header=None
)

df.columns = COLUMNS

print(df.head())
print(df.tail())

   time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23
0     0      39.25       0.43      22.75       0.43      33.75        1.3
1   250      39.25       0.43      23.00       0.00      33.00        0.0
2   500      39.25       0.43      23.25       0.43      33.00        0.0
3   750      39.50       0.50      23.00       0.71      33.00        0.0
4  1000      39.50       0.50      24.00       0.00      33.00        0.0
       time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23
475  118750      43.33       0.47      25.00       0.00       30.0        0.0
476  119000      43.50       0.50      25.50       0.50       30.0        0.0
477  119250      43.50       0.50      24.75       0.43       30.0        0.0
478  119500      43.50       0.50      24.33       0.47       30.0        0.0
479  119750      43.50       0.50      24.25       0.43       30.0        0.0


In [4]:
df.shape

(480, 7)

In [5]:
df.columns

Index(['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23',
       'var_rss23'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   time       480 non-null    int64  
 1   avg_rss12  480 non-null    float64
 2   var_rss12  480 non-null    float64
 3   avg_rss13  480 non-null    float64
 4   var_rss13  480 non-null    float64
 5   avg_rss23  480 non-null    float64
 6   var_rss23  480 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 26.4 KB


In [7]:
df.isna().sum()

time         0
avg_rss12    0
var_rss12    0
avg_rss13    0
var_rss13    0
avg_rss23    0
var_rss23    0
dtype: int64

In [8]:
all_files = sorted(DATA_DIR.glob("*/*.csv"))

inspection_results = []
read_errors = []

for file_path in all_files:
    
    relative_path = str(file_path.relative_to(DATA_DIR))
    
    try:
        current_df = pd.read_csv(
            file_path,
            comment="#",
            header=None
        )
        
        inspection_results.append({
            "file": relative_path,
            "rows": current_df.shape[0],
            "columns": current_df.shape[1],
            "missing_values": current_df.isna().sum().sum(),
        })
        
    except Exception as error:
        read_errors.append({
            "file": relative_path,
            "error_type": type(error).__name__,
            "error_message": str(error)
        })

In [9]:
inspection_df = pd.DataFrame(inspection_results)
#inspection_df

unusual_files = inspection_df[
    (inspection_df["rows"] != 480)
    | (inspection_df["columns"] != 7)
    | (inspection_df["missing_values"] > 0)
]

unusual_files

,file,rows,columns,missing_values
10,bending2/dataset4.csv,480,1,0
54,sitting/dataset8.csv,479,7,0


In [10]:
error_df = pd.DataFrame(read_errors)

error_df

#found that these datasets have more then 7 values in each row 

,file,error_type,error_message
0,cycling/dataset14.csv,ParserError,Error tokenizing data. C error: Expected 7 fie...
1,cycling/dataset9.csv,ParserError,Error tokenizing data. C error: Expected 7 fie...


In [11]:
print("Successfully read:", len(inspection_df))
print("Read errors:", len(error_df))
print("Total files:", len(all_files))

Successfully read: 86
Read errors: 2
Total files: 88


In [12]:
# clean sitting/dataset8.csv

file_path = DATA_DIR / "sitting" / "dataset8.csv"

df = pd.read_csv(
    file_path,
    comment="#",
    header=None,
    names=COLUMNS
)

expected_times = list(range(0, 120000, 250))
#len(expected_times) -> 480

actual_times = df["time"].tolist()

missing_times = sorted(
    set(expected_times) - set(actual_times)
)

print(f'time missing:{missing_times}')

set(expected_times) - set(actual_times)

df = df.set_index("time")
df = df.reindex(expected_times)
display(df.loc[[13250, 13500, 13750]])
print()

df = df.interpolate(method="linear")
display(df.loc[[13250, 13500, 13750]])

df = df.reset_index()
df = df.rename(columns={"index": "time"})
print()
print(df.shape)
print()
print(df.isna().sum())

time missing:[13500]


,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
time,,,,,,
13250,42.0,0.0,16.87,3.59,16.29,4.67
13500,NaN,NaN,NaN,NaN,NaN,NaN
13750,42.0,0.0,18.25,1.79,18.25,2.17


,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
time,,,,,,
13250,42.0,0.0,16.87,3.59,16.29,4.67
13500,42.0,0.0,17.56,2.69,17.27,3.42
13750,42.0,0.0,18.25,1.79,18.25,2.17



(480, 7)

time         0
avg_rss12    0
var_rss12    0
avg_rss13    0
var_rss13    0
avg_rss23    0
var_rss23    0
dtype: int64


In [13]:
# clean bending2/dataset4.csv

file_path = DATA_DIR / "bending2" / "dataset4.csv"

df = pd.read_csv(
    file_path,
    comment="#",
    header=None,
    names=COLUMNS
)

#df.shape -> (480,1)

display(df.head())

with open(file_path, "r") as file:
    for line in file:
        if not line.startswith("#") and line.strip():
            print(repr(line))
            break

print("Number of commas:", line.count(","))
print("Values separated by whitespace:", len(line.split()))

df = pd.read_csv(
    file_path,
    comment="#",
    header=None,
    names=COLUMNS,
    sep=r"\s+"
)

print()
print(f'shape after cleaning: {df.shape}')
display(df.head())
df.info()

,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0 32.50 0.50 0.00 0.00 19.00 1.00,NaN,NaN,NaN,NaN,NaN,NaN
1,250 32.50 0.50 0.00 0.00 18.50 0.50,NaN,NaN,NaN,NaN,NaN,NaN
2,500 32.75 0.43 1.00 0.00 18.00 0.00,NaN,NaN,NaN,NaN,NaN,NaN
3,750 32.50 0.50 0.00 0.00 17.50 0.50,NaN,NaN,NaN,NaN,NaN,NaN
4,1000 32.50 0.50 7.50 0.50 17.50 0.87,NaN,NaN,NaN,NaN,NaN,NaN


'0 32.50 0.50 0.00 0.00 19.00 1.00 \n'
Number of commas: 0
Values separated by whitespace: 7

shape after cleaning: (480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,32.50,0.50,0.0,0.0,19.0,1.00
1,250,32.50,0.50,0.0,0.0,18.5,0.50
2,500,32.75,0.43,1.0,0.0,18.0,0.00
3,750,32.50,0.50,0.0,0.0,17.5,0.50
4,1000,32.50,0.50,7.5,0.5,17.5,0.87


<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   time       480 non-null    int64  
 1   avg_rss12  480 non-null    float64
 2   var_rss12  480 non-null    float64
 3   avg_rss13  480 non-null    float64
 4   var_rss13  480 non-null    float64
 5   avg_rss23  480 non-null    float64
 6   var_rss23  480 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 26.4 KB


In [14]:
#clean cycling/dataset9.csv
#there are extra comma in the end of each row

file_path = DATA_DIR / "cycling" / "dataset9.csv"

expected_number_of_values = 7
data_row_number = 0

with open(file_path, "r") as file:
    
    for file_line_number, line in enumerate(file, start=1):
        
        if line.startswith("#") or not line.strip():
            continue
        
        data_row_number += 1
        
        values = line.strip().split(",")
        
        if len(values) != expected_number_of_values:
            print("Data row:", data_row_number)
            print("File line:", file_line_number)
            print("Number of values:", len(values))
            print("Values:", values)
            print("Original line:", repr(line))

line.strip().split(",")

cleaned_rows = []

with open(file_path, "r") as file:
    
    for line in file:
        
        if line.startswith("#") or not line.strip():
            continue
        
        line = line.strip()
        
        #keep the string from the start up to the second-to-last character to remove the trailing comma.
        if line.endswith(","):
            line = line[:-1]
        
        values = line.split(",")
        cleaned_rows.append(values)

df = pd.DataFrame(
    cleaned_rows,
    columns=COLUMNS
)

df = df.apply(pd.to_numeric)

print()
print(df.shape)
print()
print(df.dtypes)
print()
print(df.isna().sum())

Data row: 480
File line: 485
Number of values: 8
Values: ['119750', '38.33', '0.94', '15.25', '2.17', '20.33', '1.25', '']
Original line: '119750,38.33,0.94,15.25,2.17,20.33,1.25,'

(480, 7)

time           int64
avg_rss12    float64
var_rss12    float64
avg_rss13    float64
var_rss13    float64
avg_rss23    float64
var_rss23    float64
dtype: object

time         0
avg_rss12    0
var_rss12    0
avg_rss13    0
var_rss13    0
avg_rss23    0
var_rss23    0
dtype: int64


In [15]:
#clean cycling/dataset14.csv
#there are extra comma in the end of each row

file_path = DATA_DIR / "cycling" / "dataset14.csv"

expected_number_of_values = 7
data_row_number = 0

with open(file_path, "r") as file:
    
    for file_line_number, line in enumerate(file, start=1):
        
        if line.startswith("#") or not line.strip():
            continue
        
        data_row_number += 1
        
        values = line.strip().split(",")
        
        if len(values) != expected_number_of_values:
            print("Data row:", data_row_number)
            print("File line:", file_line_number)
            print("Number of values:", len(values))
            print("Values:", values)
            print("Original line:", repr(line))

line.strip().split(",")

cleaned_rows = []

with open(file_path, "r") as file:
    
    for line in file:
        
        if line.startswith("#") or not line.strip():
            continue
        
        line = line.strip()
        
        #keep the string from the start up to the second-to-last character to remove the trailing comma.
        if line.endswith(","):
            line = line[:-1]
        
        values = line.split(",")
        cleaned_rows.append(values)

df = pd.DataFrame(
    cleaned_rows,
    columns=COLUMNS
)

df = df.apply(pd.to_numeric)

print()
print(df.shape)
print()
print(df.dtypes)
print()
print(df.isna().sum())

Data row: 480
File line: 485
Number of values: 8
Values: ['119750', '34.00', '2.12', '24.50', '1.80', '22.25', '2.59', '']
Original line: '119750,34.00,2.12,24.50,1.80,22.25,2.59,'

(480, 7)

time           int64
avg_rss12    float64
var_rss12    float64
avg_rss13    float64
var_rss13    float64
avg_rss23    float64
var_rss23    float64
dtype: object

time         0
avg_rss12    0
var_rss12    0
avg_rss13    0
var_rss13    0
avg_rss23    0
var_rss23    0
dtype: int64


In [16]:
def read_comma_file(file_path):
    
    rows = []
    
    with open(file_path, "r") as file:
        
        for line in file:
            
            line = line.strip()
            
            if line.startswith("#") or not line:
                continue
            
            if line.endswith(","):
                line = line[:-1]
            
            values = line.split(",")
            
            if len(values) != 7:
                raise ValueError(
                    f"{file_path} has a row with {len(values)} values"
                )
            
            rows.append(values)
    
    df = pd.DataFrame(rows, columns=COLUMNS)
    
    df = df.apply(pd.to_numeric)
    
    return df

all_datasets = {}

expected_times = list(range(0, 120000, 250))

all_files = sorted(DATA_DIR.glob("*/*.csv"))

for file_path in all_files:
    
    # example: bending1/dataset1.csv
    file_name = file_path.relative_to(DATA_DIR).as_posix()
    
    # this file uses whitespace as its separator
    if file_name == "bending2/dataset4.csv":
        
        df = pd.read_csv(
            file_path,
            comment="#",
            header=None,
            names=COLUMNS,
            sep=r"\s+"
        )
    
    # other files use commas
    else:
        df = read_comma_file(file_path)
    
    # fill the missing timestamp and sensor values
    if file_name == "sitting/dataset8.csv":
        
        df = df.set_index("time")
        df = df.reindex(expected_times)
        df = df.interpolate(method="linear")
        df = df.reset_index()
    
    # add the cleaned DataFrame to the dictionary
    all_datasets[file_name] = df

In [17]:
print("Total instances:", len(all_datasets))

print(
    "Every instance is 480 × 7:",
    all(df.shape == (480, 7) for df in all_datasets.values())
)

print(
    "Total missing values:",
    sum(
        df.isna().sum().sum()
        for df in all_datasets.values()
    )
)

Total instances: 88
Every instance is 480 × 7: True
Total missing values: 0


### (b) Test and Train Data

In [18]:
train_datasets = {}
test_datasets = {}

split_records = []

for file_name, df in all_datasets.items():
    
    activity, dataset_file = file_name.split("/")
    
    dataset_number = int(
        dataset_file
        .replace("dataset", "")
        .replace(".csv", "")
    )
    
    # determine whether the instance belongs to the test set
    if activity in ["bending1", "bending2"]:
        is_test = dataset_number in [1, 2]
    else:
        is_test = dataset_number in [1, 2, 3]
    
    if is_test:
        test_datasets[file_name] = df.copy()
        split_name = "test"
    else:
        train_datasets[file_name] = df.copy()
        split_name = "train"
    
    split_records.append({
        "activity": activity,
        "dataset_number": dataset_number,
        "file": file_name,
        "split": split_name
    })

In [19]:
split_df = pd.DataFrame(split_records)

split_df = split_df.sort_values(
    by=["activity", "dataset_number"]
).reset_index(drop=True)

display(split_df)

split_counts = pd.crosstab(
    index=split_df["activity"],
    columns=split_df["split"],
    margins=True
)

display(split_counts)

,activity,dataset_number,file,split
0,bending1,1,bending1/dataset1.csv,test
1,bending1,2,bending1/dataset2.csv,test
2,bending1,3,bending1/dataset3.csv,train
3,bending1,4,bending1/dataset4.csv,train
4,bending1,5,bending1/dataset5.csv,train
...,...,...,...,...
83,walking,11,walking/dataset11.csv,train
84,walking,12,walking/dataset12.csv,train
85,walking,13,walking/dataset13.csv,train
86,walking,14,walking/dataset14.csv,train


split,test,train,All
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


In [20]:
for file_name in test_datasets:
    print(file_name)

print("Training instances:", len(train_datasets))
print("Test instances:", len(test_datasets))
print(
    "Total instances:",
    len(train_datasets) + len(test_datasets)
)

bending1/dataset1.csv
bending1/dataset2.csv
bending2/dataset1.csv
bending2/dataset2.csv
cycling/dataset1.csv
cycling/dataset2.csv
cycling/dataset3.csv
lying/dataset1.csv
lying/dataset2.csv
lying/dataset3.csv
sitting/dataset1.csv
sitting/dataset2.csv
sitting/dataset3.csv
standing/dataset1.csv
standing/dataset2.csv
standing/dataset3.csv
walking/dataset1.csv
walking/dataset2.csv
walking/dataset3.csv
Training instances: 69
Test instances: 19
Total instances: 88


In [21]:
assert len(train_datasets) == 69
assert len(test_datasets) == 19

# No file should appear in both sets
assert set(train_datasets).isdisjoint(set(test_datasets))

# Every original file should appear in one of the two sets
assert (
    set(train_datasets) | set(test_datasets)
    == set(all_datasets)
)

### (c) Feature Extraction

#### i. Research

Common time-domain features include the minimum,
maximum, mean, median, standard deviation, variance, range, first quartile,
third quartile, interquartile range, skewness, kurtosis, root mean square,
zero-crossing rate, and autocorrelation.

#### ii. Extraction

In [22]:
example_df = all_datasets["bending1/dataset1.csv"]

series = example_df["avg_rss12"]

print("Minimum:", series.min())
print("Maximum:", series.max())
print("Mean:", series.mean())
print("Median:", series.median())
print("Standard deviation:", series.std())
print("First quartile:", series.quantile(0.25))
print("Third quartile:", series.quantile(0.75))

Minimum: 37.25
Maximum: 45.0
Mean: 40.62479166666667
Median: 40.5
Standard deviation: 1.4769669162668115
First quartile: 39.25
Third quartile: 42.0


In [23]:
SENSOR_COLUMNS = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

series_mapping = pd.DataFrame({
    "time_series_number": range(1, 7),
    "column_name": SENSOR_COLUMNS
})

display(series_mapping)

,time_series_number,column_name
0,1,avg_rss12
1,2,var_rss12
2,3,avg_rss13
3,4,var_rss13
4,5,avg_rss23
5,6,var_rss23


In [24]:
feature_rows = []
file_names = []

for file_name in sorted(all_datasets):
    
    df = all_datasets[file_name]
    row_features = {}
    
    for series_number, column_name in enumerate(
        SENSOR_COLUMNS,
        start=1
    ):
        series = df[column_name]
        
        row_features[f"min{series_number}"] = series.min()
        row_features[f"max{series_number}"] = series.max()
        row_features[f"mean{series_number}"] = series.mean()
        row_features[f"median{series_number}"] = series.median()
        row_features[f"std{series_number}"] = series.std()
        
        row_features[
            f"first_quartile{series_number}"
        ] = series.quantile(0.25)
        
        row_features[
            f"third_quartile{series_number}"
        ] = series.quantile(0.75)
    
    feature_rows.append(row_features)
    file_names.append(file_name)

feature_df = pd.DataFrame(
    feature_rows,
    index=file_names
)

feature_df.index.name = "file"
display(feature_df.head())

,min1,max1,mean1,median1,std1,first_quartile1,third_quartile1,min2,max2,mean2,...,std5,first_quartile5,third_quartile5,min6,max6,mean6,median6,std6,first_quartile6,third_quartile6
file,,,,,,,,,,,,,,,,,,,,,
bending1/dataset1.csv,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
bending1/dataset2.csv,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
bending1/dataset3.csv,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
bending1/dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
bending1/dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [25]:
feature_df.columns

Index(['min1', 'max1', 'mean1', 'median1', 'std1', 'first_quartile1',
       'third_quartile1', 'min2', 'max2', 'mean2', 'median2', 'std2',
       'first_quartile2', 'third_quartile2', 'min3', 'max3', 'mean3',
       'median3', 'std3', 'first_quartile3', 'third_quartile3', 'min4', 'max4',
       'mean4', 'median4', 'std4', 'first_quartile4', 'third_quartile4',
       'min5', 'max5', 'mean5', 'median5', 'std5', 'first_quartile5',
       'third_quartile5', 'min6', 'max6', 'mean6', 'median6', 'std6',
       'first_quartile6', 'third_quartile6'],
      dtype='str')

In [26]:
assert feature_df.shape == (88, 42)
assert feature_df.isna().sum().sum() == 0

print("Feature extraction completed successfully.")

Feature extraction completed successfully.


In [27]:
train_feature_df = feature_df.loc[
    list(train_datasets.keys())
]

test_feature_df = feature_df.loc[
    list(test_datasets.keys())
]

print("Training feature shape:", train_feature_df.shape)
print("Test feature shape:", test_feature_df.shape)

Training feature shape: (69, 42)
Test feature shape: (19, 42)


In [28]:
# normalize/standardize experiment

from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

train_feature_normalized = pd.DataFrame(
    minmax_scaler.fit_transform(train_feature_df),
    index=train_feature_df.index,
    columns=train_feature_df.columns
)

test_feature_normalized = pd.DataFrame(
    minmax_scaler.transform(test_feature_df),
    index=test_feature_df.index,
    columns=test_feature_df.columns
)

In [29]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

train_feature_standardized = pd.DataFrame(
    standard_scaler.fit_transform(train_feature_df),
    index=train_feature_df.index,
    columns=train_feature_df.columns
)

test_feature_standardized = pd.DataFrame(
    standard_scaler.transform(test_feature_df),
    index=test_feature_df.index,
    columns=test_feature_df.columns
)

#### iii. Standard Deviation

In [30]:
feature_standard_deviations = feature_df.std()

display(feature_standard_deviations)

min1               9.569975
max1               4.394362
mean1              5.335703
median1            5.440054
std1               1.772155
first_quartile1    6.153590
third_quartile1    5.138925
min2               0.000000
max2               5.062729
mean2              1.574170
median2            1.412244
std2               0.884104
first_quartile2    0.946386
third_quartile2    2.125266
min3               2.956462
max3               4.875137
mean3              4.008386
median3            4.036396
std3               0.946695
first_quartile3    4.220658
third_quartile3    4.171628
min4               0.000000
max4               2.183625
mean4              1.166071
median4            1.145586
std4               0.458200
first_quartile4    0.843620
third_quartile4    1.552504
min5               6.124001
max5               5.741238
mean5              5.675593
median5            5.813782
std5               1.024873
first_quartile5    6.096465
third_quartile5    5.531720
min6               0

In [31]:
#bootstrap sample
bootstrap_sample = feature_df.sample(
    n=len(feature_df),
    replace=True,
    random_state=42
)

print(bootstrap_sample.shape)

bootstrap_sample.std()

(88, 42)


min1               9.422875
max1               3.551549
mean1              4.916287
median1            5.000480
std1               1.636385
first_quartile1    5.788235
third_quartile1    4.480649
min2               0.000000
max2               5.296109
mean2              1.613928
median2            1.461466
std2               0.893482
first_quartile2    0.991674
third_quartile2    2.174643
min3               3.056836
max3               4.112278
mean3              3.785233
median3            3.938522
std3               0.800825
first_quartile3    3.967931
third_quartile3    3.877527
min4               0.000000
max4               2.066477
mean4              1.167452
median4            1.158227
std4               0.453320
first_quartile4    0.845666
third_quartile4    1.548337
min5               7.387576
max5               5.660480
mean5              6.578155
median5            6.808052
std5               0.851171
first_quartile5    6.941676
third_quartile5    6.407483
min6               0

In [32]:
np.random.seed(42)

number_of_bootstraps = 10000
bootstrap_standard_deviations = []

for i in range(number_of_bootstraps):
    
    bootstrap_sample = feature_df.sample(
        n=len(feature_df),
        replace=True
    )
    
    sample_standard_deviations = bootstrap_sample.std()
    
    bootstrap_standard_deviations.append(
        sample_standard_deviations
    )

bootstrap_std_df = pd.DataFrame(
    bootstrap_standard_deviations
)

bootstrap_std_df.shape

(10000, 42)

In [33]:
lower_bounds = bootstrap_std_df.quantile(0.05)
upper_bounds = bootstrap_std_df.quantile(0.95)

bootstrap_results = pd.DataFrame({
    "Standard Deviation": feature_standard_deviations,
    "90% CI Lower Bound": lower_bounds,
    "90% CI Upper Bound": upper_bounds
})

bootstrap_results.index.name = "Feature"

display(bootstrap_results.round(3))

,Standard Deviation,90% CI Lower Bound,90% CI Upper Bound
Feature,,,
min1,9.570,8.265,10.756
max1,4.394,3.341,5.307
mean1,5.336,4.714,5.874
median1,5.440,4.787,5.995
std1,1.772,1.569,1.954
first_quartile1,6.154,5.566,6.649
third_quartile1,5.139,4.336,5.849
min2,0.000,0.000,0.000
max2,5.063,4.622,5.405


In [34]:
inside_interval = (
    bootstrap_results["Standard Deviation"]
    >= bootstrap_results["90% CI Lower Bound"]
) & (
    bootstrap_results["Standard Deviation"]
    <= bootstrap_results["90% CI Upper Bound"]
)

print(
    "Number of estimates inside their intervals:",
    inside_interval.sum()
)

# assert bootstrap_results.shape == (42, 3)
# assert bootstrap_results.isna().sum().sum() == 0
# assert all(
#     bootstrap_results["90% CI Lower Bound"]
#     <= bootstrap_results["90% CI Upper Bound"]
# )

Number of estimates inside their intervals: 42


#### iv. Select Features

I selected the minimum, maximum, and mean as the three most important
time-domain feature types.

The mean represents the overall level of a time series, while the minimum
and maximum represent its lower and upper extremes. Together, these features
provide information about both the central tendency and the range of sensor
measurements for each activity. They are also straightforward to interpret
and may help distinguish activities with different average levels and
movement ranges.

## 2. ISLR 3.7.4

### (a) Linear Train

Cubic regression will have a training RSS that is less than or equal to
that of the linear regression.

The cubic model contains the linear model as a special case when the
coefficients of \(X^2\) and \(X^3\) are both zero. Therefore, the cubic model
can always fit the training data at least as well as the linear model.
Because it has additional flexibility, it will typically achieve a slightly
lower training RSS by fitting some of the random noise, even though the true
relationship is linear.

### (b) Linear Test

Linear regression is expected to have a lower test RSS.

Since the true relationship between \(X\) and \(Y\) is linear, the linear
model has the correct functional form. The additional \(X^2\) and \(X^3\)
terms in the cubic model do not provide a true reduction in bias. Instead,
they may fit random noise in the training data and increase the variance of
the fitted model. Therefore, although the cubic model has a training RSS
that is no greater than that of the linear model, it is expected to have a
higher test RSS due to overfitting.

### (c) Not Linear Train

Cubic regression will have a training RSS that is less than or equal to
that of the linear regression.

The cubic model contains the linear model as a special case, so its
additional flexibility cannot increase the training RSS. Since the true
relationship is not linear, the quadratic and cubic terms may also capture
some of the nonlinear structure that the linear model cannot represent.
Therefore, the cubic model will likely have a lower training RSS, although
the size of the reduction depends on the unknown form and degree of
nonlinearity.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower
test RSS.

The cubic model is more flexible and may have lower bias if the true
relationship contains substantial nonlinear structure that can be
approximated by a cubic function. In that case, the cubic model may have the
lower test RSS. However, the cubic model also has higher variance and may
overfit the training data. If the true relationship is only slightly
nonlinear, or if its nonlinearity is not well approximated by a cubic
function, the simpler linear model may have the lower test RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 